In [ ]:
#@title Import
import torch , torch.nn as nn , torch.nn.functional as F , torchvision
from torch.utils.data import DataLoader
from torchvision import transforms , datasets, models
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
#@title Device
# CHANGED: use a GPU runtime (Runtime > Change runtime type > T4 GPU) - training the teacher + student is slow on CPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)


In [ ]:
#@title Variables
BATCH_SIZE = 256
IMG_SIZE = 32          # CHANGED: CIFAR-10 images are 32x32 (MNIST was 28x28)
PATCH_SIZE = 4         # CHANGED: 32/4 = 8 -> 8x8 = 64 patches (MNIST had 16 patches)
ATTENTION_HEADS = 4
TRANSFORMER_LAYER = 6  # CHANGED: CIFAR-10 is harder than MNIST so the student needs a bit more capacity
EMBED_DIM = 128        # CHANGED: 32 was too small for CIFAR-10
CLASSES = 10
EPOCHS_TEACHER = 3     # NEW: the teacher must be fine-tuned on the dataset first
LR_TEACHER = 1e-4      # NEW
EPOCHS_STUDENT = 30    # CHANGED: ViTs need more epochs on CIFAR-10
LR_STUDENT = 5e-4      # CHANGED
CHANNELS = 3           # CIFAR-10 is already RGB, so no need to expand 1 channel -> 3 channels anymore
TEMPERATURE = 2
ALPHA = 0.5
TEACHER_IMG = 64       # NEW: teacher sees the image upsampled to 64x64 (ResNet downsamples a lot, 32x32 is too small for it)


In [ ]:
#@title Import the Data
# CHANGED: MNIST -> CIFAR10. Removed the .repeat(3,1,1) lambda (CIFAR already has 3 channels)
# NEW: Normalize + simple augmentation (random crop + flip) for train set. Augmentation matters a lot for ViT on CIFAR.
mean, std = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)

train_transformations = transforms.Compose([transforms.RandomCrop(32, padding=4),
                                            transforms.RandomHorizontalFlip(),
                                            transforms.ToTensor(),
                                            transforms.Normalize(mean, std)])
val_transformations = transforms.Compose([transforms.ToTensor(),
                                          transforms.Normalize(mean, std)])

train_dataset = datasets.CIFAR10('./data',train=True, download=True, transform= train_transformations)
val_dataset = datasets.CIFAR10('./data',train=False, download=True,  transform= val_transformations)


In [ ]:
#@title Create train and val batches
train_dataloader = DataLoader(train_dataset,batch_size= BATCH_SIZE, shuffle=True, num_workers=2)
val_dataloader = DataLoader(val_dataset,batch_size= BATCH_SIZE, num_workers=2)


In [ ]:
#@title Teacher Model
teacher = models.resnet50(weights= models.ResNet50_Weights.IMAGENET1K_V2)
# we have to remove last layer i.e 1000 class classification bcs we want it to predict just 10 class of cifar10 dataset
teacher.fc = nn.Linear(teacher.fc.in_features,CLASSES)
teacher = teacher.to(device)

# NEW: helper that upsamples 32x32 -> 64x64 before giving the image to the teacher
def teacher_forward(x):
  x = F.interpolate(x, size=TEACHER_IMG, mode='bilinear', align_corners=False)
  return teacher(x)


In [ ]:
#@title Evaluate function
# NEW: one reusable function so we can evaluate the teacher, the distilled student and the baseline the same way
# kind = 'teacher' -> teacher accuracy
# kind = 'cls'     -> only the class-token head
# kind = 'dist'    -> only the distillation-token head
# kind = 'avg'     -> average of both heads (what you did in your MNIST code)
def evaluate(model, kind):
  model.eval()
  correct=0; total=0
  with torch.no_grad():
    for x,y in val_dataloader:
      x,y = x.to(device), y.to(device)
      if kind == 'teacher':
        out = teacher_forward(x)
      else:
        cls_logits, dist_logits = model(x)
        if kind == 'cls': out = cls_logits
        elif kind == 'dist': out = dist_logits
        else: out = (cls_logits + dist_logits)/2
      correct += (out.argmax(1)==y).sum().item(); total += y.size(0)
  return 100 * correct/total


In [ ]:
#@title Fine-tune the Teacher
# NEW: in the MNIST code the new teacher.fc layer was never trained, so the teacher was giving random outputs.
# A teacher has to be good at the task first, so we fine-tune it for a few epochs.
opt_t = torch.optim.AdamW(teacher.parameters(), lr=LR_TEACHER)
for e in range(EPOCHS_TEACHER):
  teacher.train()
  for x,y in train_dataloader:
    x,y = x.to(device), y.to(device)
    loss = F.cross_entropy(teacher_forward(x), y)
    opt_t.zero_grad(); loss.backward(); opt_t.step()
  print(f"Teacher epoch {e+1} done | val acc: {evaluate(teacher,'teacher'):.2f}%")

teacher.eval()  # NEW: freeze BN/dropout behaviour for the teacher from now on
for p in teacher.parameters(): p.requires_grad = False


In [ ]:
#@title Student Vit Patch Embedding
# Patch Embed
# here kernel size and stride both are patch_size (suppose our patch size = 7 then our each patch will 7x7
# so whenever we want it to shift we have shift it to the same size so stride should also be 7x7 so that there would be no overlapping)
class PatchEmbed(nn.Module):
  def __init__(self,num_channels= CHANNELS, embed_dim=EMBED_DIM, patch_size= PATCH_SIZE,img_size = IMG_SIZE):
    super().__init__()
    self.proj = nn.Conv2d(num_channels ,embed_dim ,patch_size , patch_size)
    self.n = (img_size // patch_size) ** 2 # --> no.of patches or patch_num

  def forward(self,x):
    x = self.proj(x)
    x = x.flatten(2)
    x = x.transpose(1,2)
    return x


In [ ]:
#@title Student Vit Class
# qkv_dim is nothing but dimension of each individual attention head
# * is used to repeat the line of code
# head_cls & head_distil is similar as the mlp head class we had in vit file
# they both are going used by the model to predict and calculate the loss
# BATCH_SIZE may varies for some batches in the end so its good to extract it first
class ViT(nn.Module):
  # CHANGED: added distill=True. With distill=False the model has NO distillation token and NO distillation head,
  # so it becomes a normal ViT (only cls token). This is our baseline for comparison.
  def __init__(self,embed_dim= EMBED_DIM, attention_heads= ATTENTION_HEADS, layers= TRANSFORMER_LAYER,classes = CLASSES, distill=True):
    super().__init__()
    self.distill = distill
    self.patch_embed = PatchEmbed()
    self.cls = nn.Parameter(torch.zeros(1,1,embed_dim))
    n = self.patch_embed.n
    if distill:
      self.distillation = nn.Parameter(torch.zeros(1,1,embed_dim))
      self.head_distil = nn.Linear(embed_dim, classes)
      self.position = nn.Parameter(torch.zeros(1,n+2,embed_dim))  # patches + cls + dist
    else:
      self.position = nn.Parameter(torch.zeros(1,n+1,embed_dim))  # NEW: patches + cls only
    qkv_dim = embed_dim // attention_heads
    # CHANGED: the 3rd argument of TransformerEncoderLayer is dim_feedforward (the MLP size), not the head dim.
    # You had it as qkv_dim (=8) which made the MLP tiny. Standard ViT uses 4 * embed_dim.
    self.transformer_block = nn.Sequential(*[nn.TransformerEncoderLayer(embed_dim,attention_heads,4*embed_dim , batch_first=True) for _ in range(layers)])

    self.layernorm = nn.LayerNorm(embed_dim)
    self.head_cls = nn.Linear(embed_dim, classes)

  def forward(self,x):
    B = x.shape[0]
    x = self.patch_embed(x)
    cls = self.cls.expand(B,-1,-1)
    if self.distill:
      dist = self.distillation.expand(B,-1,-1)
      x = torch.cat([cls,x,dist],1) + self.position # cls + 64 patches + dist, dim = 1 bcs 0 is batch_num and 1 is num_patches and we want out cls to concat in num_patches
    else:
      x = torch.cat([cls,x],1) + self.position      # NEW: cls + 64 patches (no dist token)
    x = self.transformer_block(x)
    x = self.layernorm(x)

    cls_token = x[:,0] # for every batch 1 token i.e first token
    cls_logits = self.head_cls(cls_token)

    if not self.distill:
      return cls_logits, None  # NEW: baseline has no distillation output

    distil_token = x[:,-1] # for every batch 1 token i.e last token
    dist_logits = self.head_distil(distil_token)

    return cls_logits, dist_logits


In [ ]:
#@title Define Student Model
student = ViT(distill=True).to(device)
opt_s = torch.optim.AdamW(student.parameters(), lr= LR_STUDENT)


In [ ]:
#@title Define the loss function
# kl divergence loss
def kd_loss(cls_logits, distil_logits, t_logits, y, alpha=ALPHA, temperature = TEMPERATURE):
  kd = F.kl_div(F.log_softmax(distil_logits/temperature, dim=1),
                F.softmax(t_logits/temperature, dim=1),
                reduction = 'batchmean') * (temperature**2)
  ce = F.cross_entropy(cls_logits, y)
  return alpha * kd + (1-alpha) * ce


In [ ]:
#@title Run Epochs (Distilled Student)
# (removed the unused "from math import dist" line)
for e in range(EPOCHS_STUDENT):
  student.train()
  for x,y in train_dataloader:
    x,y = x.to(device), y.to(device)
    with torch.no_grad(): t_logits = teacher_forward(x)  # CHANGED: teacher gets the upsampled image
    cls_logits, dist_logits = student(x)
    loss = kd_loss(cls_logits, dist_logits, t_logits, y)
    opt_s.zero_grad(); loss.backward(); opt_s.step()
  # NEW: print val accuracy every epoch so you can see the learning curve
  print(f"Epochs {e+1} done | val acc (avg of heads): {evaluate(student,'avg'):.2f}%")


In [ ]:
#@title Evaluate Distilled Student
# CHANGED: evaluation now uses the evaluate() function; we report each head separately and the average
acc_teacher = evaluate(teacher, 'teacher')
acc_cls  = evaluate(student, 'cls')
acc_dist = evaluate(student, 'dist')
acc_avg  = evaluate(student, 'avg')
print(f"Teacher accuracy:                 {acc_teacher:.2f}%")
print(f"Distilled student (cls head):     {acc_cls:.2f}%")
print(f"Distilled student (dist head):    {acc_dist:.2f}%")
print(f"Distilled student (cls+dist avg): {acc_avg:.2f}%")


In [ ]:
#@title Baseline ViT (no distillation token)
# NEW: same ViT, same size, same epochs / lr, but distill=False -> no distillation token, no teacher, only cross entropy loss
baseline = ViT(distill=False).to(device)
opt_b = torch.optim.AdamW(baseline.parameters(), lr= LR_STUDENT)

for e in range(EPOCHS_STUDENT):
  baseline.train()
  for x,y in train_dataloader:
    x,y = x.to(device), y.to(device)
    cls_logits, _ = baseline(x)
    loss = F.cross_entropy(cls_logits, y)   # only hard labels, teacher is not used at all
    opt_b.zero_grad(); loss.backward(); opt_b.step()
  print(f"Baseline epoch {e+1} done | val acc: {evaluate(baseline,'cls'):.2f}%")


In [ ]:
#@title Compare
# NEW: final comparison of everything
acc_base = evaluate(baseline, 'cls')
print(f"Teacher (ResNet50):                 {acc_teacher:.2f}%")
print(f"Student ViT with distillation token: {acc_avg:.2f}%")
print(f"Baseline ViT (cls token only):       {acc_base:.2f}%")
print(f"Gain from distillation:              {acc_avg - acc_base:+.2f}%")
